In [ ]:
import marimo as mo

In [ ]:
import polars as pl
import numpy as np

# Data preprocessing

## Class definition
- 0: OK
- 1: TOXIC
- 2: SPAM

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
df = pl.read_parquet("data/encoded_df.parquet")
df.sample(5)

In [ ]:
df.describe()

## Class Distribution

In [ ]:
df["label"].value_counts()

## Train / Test split

In [ ]:
y = df.drop_in_place("label")
X = df.clone()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y)

X_train = X_train["text"].to_list()
X_test = X_test["text"].to_list()

## Train Baseline SVM

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

tfidf_vectorizer = TfidfVectorizer()

svc = LinearSVC(max_iter=1000, class_weight="balanced")

In [ ]:
pipeline = Pipeline([
    ("tf-idf", tfidf_vectorizer),
    ("svc", svc)
])

In [ ]:
pipeline.fit(X_train, y_train)

## Test baseline

In [ ]:
from sklearn.metrics import classification_report

In [ ]:
y_pred = pipeline.predict(X_test)

In [ ]:
print(classification_report(y_test, y_pred))

In [ ]:
class_mapping = {0: "OK", 1: "TOXIC", 2: "SPAM"}

In [ ]:
test_cases = [
    "отличный товар рекомендую всем пять звезд",
    "купи дешевле на сайте xxx.ru акция до конца месяца",
    "продавец м***к верните деньги уроды",
    "хороший товар хороший товар хороший товар хороший товар",
    "товар не пришел очень расстроена",
    "Блузка хорошая, пошив просто класс",
    "Дабуди дабудай"
]

predictions = pipeline.predict(test_cases)
for text, pred in zip(test_cases, predictions):
    print(f"{class_mapping[pred]} | {text}")